# Multi-Symbol Binance Klines Example

This notebook shows how to call the local `binance_klines_data_fetch` package from an external Jupyter environment and fetch closed 1-minute klines for multiple Binance USD-M Futures symbols.

Run the cleanup cell at the end when you are done, because `MultiSymbolKlineService` starts a background thread.

## 1. Add The Project To `sys.path`

Use this when the package has not been installed with `pip install -e`.

In [1]:
import sys
from pathlib import Path

repo_path = Path("/home/suncong/binance_klines_data_fetch")
repo_path_str = str(repo_path)

if repo_path_str not in sys.path:
    sys.path.insert(0, repo_path_str)

from IPython.display import display
from binance_klines_data_fetch import MultiSymbolKlineService

print("Imported from:", repo_path)

Imported from: /home/suncong/binance_klines_data_fetch


## 2. Start Multi-Symbol Background Service

This cell accesses the real Binance REST API. The service fetches closed `1m` candles only and keeps each symbol in its own rolling pandas DataFrame cache.

In [ ]:
symbols = ["BTCUSDT", "ETHUSDT", "SOLUSDT"]

service = MultiSymbolKlineService(
    symbols=symbols,
    window_size=500, #表示每个 symbol 最多缓存最近 500 根已收盘的 1 分钟 K 线
    max_workers=4, #表示最大并发请求数为 4
    refresh_interval_seconds=2.0, #表示每隔 2 秒刷新一次缓存，也就是后台线程每隔 2 秒检查一次是否有新的已收盘 1 分钟 K 线
    startup_timeout_seconds=60.0, #表示启动服务时等待的最大时间，如果超过 60 秒还没有准备好，则抛出异常 （如果symbols比较多的话 可以设置的大一点）
)

service.start(block_until_ready=True, timeout=60.0)
print("service ready:", service.status().ready)

service ready: True


## 3. Read One Symbol

In [4]:
btc_df = service.get_recent("BTCUSDT", 100)

print("rows:", len(btc_df))
print("last open time:", btc_df.index[-1])
display(btc_df.tail(20))

rows: 100
last open time: 2026-05-30 07:16:00+00:00


,Open,High,Low,Close,Volume,Close_Time,Quote_Asset_Volume,Number_of_Trades,Taker_Buy_Base_Asset_Volume,Taker_Buy_Quote_Asset_Volume
Open_Time,,,,,,,,,,
2026-05-30 06:57:00+00:00,73536.4,73547.5,73515.0,73545.9,77.890,2026-05-30 06:57:59.999000+00:00,5.727263e+06,1683,27.872,2.049614e+06
2026-05-30 06:58:00+00:00,73545.9,73558.1,73545.9,73552.6,73.348,2026-05-30 06:58:59.999000+00:00,5.394616e+06,759,65.054,4.784563e+06
2026-05-30 06:59:00+00:00,73552.6,73556.0,73550.7,73550.7,20.392,2026-05-30 06:59:59.999000+00:00,1.499904e+06,482,5.337,3.925563e+05
2026-05-30 07:00:00+00:00,73550.7,73555.0,73550.7,73554.9,15.252,2026-05-30 07:00:59.999000+00:00,1.121842e+06,305,9.362,6.886062e+05
2026-05-30 07:01:00+00:00,73555.0,73555.0,73545.3,73545.4,17.908,2026-05-30 07:01:59.999000+00:00,1.317168e+06,532,4.857,3.572562e+05
2026-05-30 07:02:00+00:00,73545.4,73545.4,73515.5,73520.2,47.748,2026-05-30 07:02:59.999000+00:00,3.510784e+06,1106,13.294,9.773859e+05
2026-05-30 07:03:00+00:00,73520.3,73520.3,73511.5,73511.6,17.557,2026-05-30 07:03:59.999000+00:00,1.290746e+06,511,6.276,4.614112e+05
2026-05-30 07:04:00+00:00,73511.6,73511.6,73505.0,73505.0,19.738,2026-05-30 07:04:59.999000+00:00,1.450910e+06,843,1.293,9.504376e+04
2026-05-30 07:05:00+00:00,73505.1,73505.1,73501.5,73501.6,14.887,2026-05-30 07:05:59.999000+00:00,1.094246e+06,390,1.484,1.090771e+05


## 4. Read All Symbols

In [5]:
all_data = service.get_all_recent(100)

for symbol, df in all_data.items():
    last_open_time = df.index[-1] if not df.empty else None
    last_close = float(df["Close"].iloc[-1]) if not df.empty else None
    print(symbol, "rows=", len(df), "last_open_time=", last_open_time, "last_close=", last_close)
    display(df.tail(3))

BTCUSDT rows= 100 last_open_time= 2026-05-30 07:16:00+00:00 last_close= 73520.7


,Open,High,Low,Close,Volume,Close_Time,Quote_Asset_Volume,Number_of_Trades,Taker_Buy_Base_Asset_Volume,Taker_Buy_Quote_Asset_Volume
Open_Time,,,,,,,,,,
2026-05-30 07:14:00+00:00,73517.5,73517.5,73507.0,73507.0,12.875,2026-05-30 07:14:59.999000+00:00,9.464842e+05,514,1.397,102703.7092
2026-05-30 07:15:00+00:00,73507.0,73530.8,73507.0,73512.7,23.346,2026-05-30 07:15:59.999000+00:00,1.716377e+06,1108,5.502,404483.4923
2026-05-30 07:16:00+00:00,73512.7,73529.0,73512.6,73520.7,17.878,2026-05-30 07:16:59.999000+00:00,1.314459e+06,636,7.670,563915.8423


ETHUSDT rows= 100 last_open_time= 2026-05-30 07:16:00+00:00 last_close= 2014.83


,Open,High,Low,Close,Volume,Close_Time,Quote_Asset_Volume,Number_of_Trades,Taker_Buy_Base_Asset_Volume,Taker_Buy_Quote_Asset_Volume
Open_Time,,,,,,,,,,
2026-05-30 07:14:00+00:00,2014.89,2014.97,2014.88,2014.89,226.387,2026-05-30 07:14:59.999000+00:00,456153.02070,395,85.928,173137.67576
2026-05-30 07:15:00+00:00,2014.88,2015.42,2014.75,2014.99,434.640,2026-05-30 07:15:59.999000+00:00,875858.70007,985,171.456,345487.92585
2026-05-30 07:16:00+00:00,2014.99,2015.52,2014.82,2014.83,338.910,2026-05-30 07:16:59.999000+00:00,682992.62995,1169,160.068,322566.61610


SOLUSDT rows= 100 last_open_time= 2026-05-30 07:16:00+00:00 last_close= 82.34


,Open,High,Low,Close,Volume,Close_Time,Quote_Asset_Volume,Number_of_Trades,Taker_Buy_Base_Asset_Volume,Taker_Buy_Quote_Asset_Volume
Open_Time,,,,,,,,,,
2026-05-30 07:14:00+00:00,82.32,82.33,82.31,82.31,1361.45,2026-05-30 07:14:59.999000+00:00,1.120808e+05,218,640.64,52743.8575
2026-05-30 07:15:00+00:00,82.31,82.36,82.30,82.31,5740.03,2026-05-30 07:15:59.999000+00:00,4.726352e+05,668,2876.11,236823.1284
2026-05-30 07:16:00+00:00,82.32,82.38,82.31,82.34,13958.83,2026-05-30 07:16:59.999000+00:00,1.149566e+06,774,12006.04,988742.7312


## 5. Inspect Service And Rate Limit Status

In [6]:
status = service.status()

print("running:", status.running)
print("ready:", status.ready)
print("last_refresh_at:", status.last_refresh_at)
print("rate_limiter:", status.rate_limiter)

for symbol, item in status.symbols.items():
    print(symbol, "ready=", item.ready, "rows=", item.row_count, "last_open_time=", item.last_open_time, "error=", item.last_error)

running: True
ready: True
last_refresh_at: 2026-05-30 07:17:39.849822+00:00
rate_limiter: RateLimiterStatus(limit=2400, effective_limit=2400, window_seconds=60.0, used_weight=11, available_weight=2389, cooldown_seconds=0.0, observed_used_weight_1m=61, last_rate_limited_at=None, last_banned_at=None)
BTCUSDT ready= True rows= 500 last_open_time= 2026-05-30 07:16:00+00:00 error= None
ETHUSDT ready= True rows= 500 last_open_time= 2026-05-30 07:16:00+00:00 error= None
SOLUSDT ready= True rows= 500 last_open_time= 2026-05-30 07:16:00+00:00 error= None


## 6. Stop Background Thread

Run this before closing the notebook or before creating another service instance.

In [ ]:
service.stop()
print("service running:", service.status().running)